In [ ]:
!pip install -q google-generativeai

In [ ]:
%%writefile orchestrator.py
import os
import subprocess
import google.generativeai as genai

# Standard way to get API key from system environment
genai.configure(api_key=os.environ.get("GEMINI_API_KEY"))
model = genai.GenerativeModel('gemini-3.5-flash')

def generate_and_heal(task_description, max_retries=3):
    system_prompt = """You are an elite bare-metal C++ kernel generator.
Your ONLY job is to output raw, syntactically perfect C++ code.
RULES:
1. NO markdown tags (like ```cpp).
2. NO explanations, NO greetings, NO text outside the code.
3. Output MUST be strictly and directly compilable by g++."""

    current_task = task_description

    for attempt in range(max_retries):
        print(f"\n[Attempt {attempt + 1}/{max_retries}] Generating C++ code...")

        response = model.generate_content(f"{system_prompt}\n\nTask: {current_task}")
        raw_code = response.text.replace("```cpp", "").replace("```c++", "").replace("```", "").strip()

        with open("dynamic_hook.cpp", "w") as f:
            f.write(raw_code)

        print("Compiling with g++...")
        compile_process = subprocess.run(
            ["g++", "dynamic_hook.cpp", "-o", "dynamic_hook_exe"],
            capture_output=True,
            text=True
        )

        if compile_process.returncode == 0:
            print("✅ Compilation Successful on attempt", attempt + 1)
            return True
        else:
            print("❌ Compilation Failed! Capturing error log...")
            error_output = compile_process.stderr
            print(f"Compiler Error:\n{error_output}")

            print("🔄 Feeding error back to AI for self-healing...")
            current_task = f"{task_description}\n\nPrevious code failed to compile with this error:\n{error_output}\nFix the syntax/logic error and provide correct code."

    print("🚨 Max retries reached. Self-healing failed.")
    return False

# TRICK TASK: Forcing the AI to make a mistake on Attempt 1
task = """Write a C++ program that prints 'Boss Fight Defeated: System Healed!'.
CRITICAL INSTRUCTION FOR ATTEMPT 1: You MUST deliberately omit the '#include <iostream>' header so the code fails to compile.
When the orchestrator feeds you the compiler error back in the next attempt, fix it by adding the header and output the correct code."""

success = generate_and_heal(task)

In [ ]:
import os
import time
import subprocess
from google.colab import userdata

# 🔒 Fetch secret strictly in the notebook environment and inject it globally
os.environ["GEMINI_API_KEY"] = userdata.get('GEMINI_API_KEY')

print("--- Step 1: Running Self-Healing Orchestrator ---")
!python orchestrator.py

print("\n--- Step 2: Executing Live System Hook ---")
if os.path.exists("dynamic_hook_exe"):
    execution_result = subprocess.run(["./dynamic_hook_exe"], capture_output=True, text=True)
    print(execution_result.stdout.strip())
else:
    print("Error: Executable not found. Orchestrator failed to heal.")

print("\n--- Step 3: Zero-Footprint Cleanup Protocol ---")
time.sleep(1)

if os.path.exists("dynamic_hook.cpp"):
    os.remove("dynamic_hook.cpp")
    print("🗑️ Source code (dynamic_hook.cpp) securely deleted.")

if os.path.exists("dynamic_hook_exe"):
    os.remove("dynamic_hook_exe")
    print("🗑️ Executable binary (dynamic_hook_exe) securely deleted.")

print("👻 Liquid Kernel execution complete. System traces vaporized.")